# Update a MODFLOW WEL file with FloPy

This notebook shows how to load a MODFLOW-2005 WEL file, update pumping rates, and write an MF6 WEL package using FloPy.


In [ ]:
# If needed, install flopy in this environment
# !pip install flopy


## 1) Download the WEL file


In [ ]:
import urllib.request

url = (
    "https://ckan.tacc.utexas.edu/dataset/18400624-423c-42b5-ad56-6c73322584bd/"
    "resource/9c7b25c4-8cea-4965-a07a-d9b3867f18a9/"
    "download/barton_springs_2001_2010average.wel"
)
wel_path = "barton_springs_2001_2010average.wel"
urllib.request.urlretrieve(url, wel_path)
print("Downloaded", wel_path)


## 2) Load the WEL file with FloPy (MODFLOW-2005)

We need a minimal DIS to load the WEL. We can scan the file to infer max row/col/layer.


In [ ]:
from pathlib import Path
import flopy

def scan_wel_metadata(path):
    def strip_comment(line):
        for token in ("#", ";"):
            if token in line:
                line = line.split(token, 1)[0]
        return line.strip()

    lines = [strip_comment(line) for line in Path(path).read_text().splitlines()]
    data_lines = [line for line in lines if line]
    if not data_lines:
        raise ValueError("WEL file is empty or has no data.")
    data_lines.pop(0)  # header

    nper = 0
    max_k = max_i = max_j = 1
    idx = 0
    while idx < len(data_lines):
        tokens = data_lines[idx].split()
        idx += 1
        if not tokens:
            continue
        nper += 1
        itmp = int(tokens[0])
        if itmp <= 0:
            continue
        for _ in range(itmp):
            if idx >= len(data_lines):
                raise ValueError("Unexpected end of file while scanning wells.")
            parts = data_lines[idx].split()
            idx += 1
            k, i, j = (int(parts[0]), int(parts[1]), int(parts[2]))
            max_k = max(max_k, k)
            max_i = max(max_i, i)
            max_j = max(max_j, j)

    return nper, max_k, max_i, max_j

nper, nlay, nrow, ncol = scan_wel_metadata(wel_path)
print("nper, nlay, nrow, ncol =", nper, nlay, nrow, ncol)

m = flopy.modflow.Modflow(modelname="wel_read", model_ws=".")
flopy.modflow.ModflowDis(
    m,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    nper=nper,
    delr=1.0,
    delc=1.0,
    top=1.0,
    botm=[0.0] * nlay,
)
wel = flopy.modflow.ModflowWel.load(wel_path, m)
wel


## 3) Update pumping rates (example: scale by 10%)


In [ ]:
import numpy as np

spd = wel.stress_period_data.data

# Inspect one stress period
first_per = spd[0]
print(first_per.dtype.names)
print(first_per[:3])

# Scale all pumping rates by 1.1
for per, recs in spd.items():
    if len(recs) == 0:
        continue
    recs["flux"] *= 1.1

wel.stress_period_data = spd
print("Updated rates")


## 4) Write an MF6 WEL package with FloPy


In [ ]:
# Convert MF2005 WEL recarray -> MF6 stress period data
mf6_spd = {}
for per, recs in spd.items():
    items = []
    for rec in recs:
        k = int(rec["k"]) - 1
        i = int(rec["i"]) - 1
        j = int(rec["j"]) - 1
        q = float(rec["flux"])
        items.append(((k, i, j), q))
    mf6_spd[per] = items

sim = flopy.mf6.MFSimulation(sim_name="wel_update", version="mf6", sim_ws=".")
flopy.mf6.ModflowTdis(
    sim,
    time_units="DAYS",
    nper=nper,
    perioddata=[(1.0, 1, 1.0)] * nper,
)
gwf = flopy.mf6.ModflowGwf(sim, modelname="gwf")
flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=1.0,
    delc=1.0,
    top=1.0,
    botm=[0.0] * nlay,
)
wel6 = flopy.mf6.ModflowGwfwel(
    gwf,
    stress_period_data=mf6_spd,
    filename="barton_springs_mf6.wel",
)
wel6.write()
print("Wrote", wel6.filename)


# Use the EBFZ grid (CSV/GDB) to select wells

This section downloads the EBFZ grid from CKAN (CSV + GDB), then lets you select wells by zone,
by cell list, or across the entire region, and update the WEL data with FloPy.


In [ ]:
import zipfile
from pathlib import Path
import urllib.request

grid_url = (
    "https://ckan.tacc.utexas.edu/dataset/18400624-423c-42b5-ad56-6c73322584bd/"
    "resource/f07a257c-1d88-4819-bd5d-a104c5e3fe5b/"
    "download/ebfz_b_grid.zip"
)
grid_zip = Path("ebfz_b_grid.zip")
grid_dir = Path("ebfz_b_grid")
grid_csv = grid_dir / "ebfz_b_grid_poly101223.csv"

if not grid_csv.exists():
    print("Downloading grid...")
    urllib.request.urlretrieve(grid_url, grid_zip)
    print("Extracting...")
    with zipfile.ZipFile(grid_zip, "r") as zf:
        zf.extractall(grid_dir)
    print("Ready:", grid_csv)
else:
    print("Using local grid CSV:", grid_csv)


In [ ]:
import pandas as pd

grid_df = pd.read_csv(grid_csv)
grid_df.head()


## Interactive selection + update

Selections apply to all stress periods in the loaded WEL file.
If you want different per-period updates, we can extend this later.


In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Make sure `wel` is defined by running the earlier WEL load cell.
spd = wel.stress_period_data.data

zone_fields = [
    "GCD_Name", "BasinName", "CountyName", "GMA", "LETTER",
    "PGMA_Name", "RA_SLD", "REG_NAME"
]
zone_fields = [f for f in zone_fields if f in grid_df.columns]

mode = widgets.Dropdown(options=["All wells", "By zone", "By cell_id list", "By row,col list"], value="All wells")
zone_field = widgets.Dropdown(options=zone_fields)
zone_values = widgets.SelectMultiple(options=[])
cell_id_text = widgets.Textarea(placeholder="1001001,1001002", description="CELL_IDs")
rowcol_text = widgets.Textarea(placeholder="1,1
1,2", description="row,col")
layer_new = widgets.IntText(value=1, description="Layer (k)")
rate_mode = widgets.Dropdown(options=["multiply", "set"], value="multiply")
rate_value = widgets.FloatText(value=1.1, description="Value")
add_missing = widgets.Checkbox(value=False, description="Add missing wells")
apply_btn = widgets.Button(description="Apply update", button_style="primary")
output = widgets.Output()

def _update_zone_values(*_):
    if zone_field.value:
        vals = sorted(grid_df[zone_field.value].dropna().unique().tolist())
        zone_values.options = vals

zone_field.observe(_update_zone_values, names="value")
_update_zone_values()

def _parse_cell_ids(text):
    items = []
    for part in text.replace('
', ',').split(','):
        part = part.strip()
        if part:
            items.append(int(part))
    return items

def _parse_rowcols(text):
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = [p.strip() for p in line.split(',')]
        if len(parts) != 2:
            raise ValueError(f"Invalid row,col: {line}")
        rows.append((int(parts[0]), int(parts[1])))
    return rows

def _selected_cells():
    if mode.value == "All wells":
        return set(zip(grid_df["ROW"], grid_df["COL"]))
    if mode.value == "By zone":
        if not zone_values.value:
            return set()
        df = grid_df[grid_df[zone_field.value].isin(list(zone_values.value))]
        return set(zip(df["ROW"], df["COL"]))
    if mode.value == "By cell_id list":
        ids = _parse_cell_ids(cell_id_text.value)
        df = grid_df[grid_df["CELL_ID"].isin(ids)]
        return set(zip(df["ROW"], df["COL"]))
    if mode.value == "By row,col list":
        return set(_parse_rowcols(rowcol_text.value))
    return set()

def _apply_update(_):
    with output:
        clear_output()
        selected = _selected_cells()
        if not selected:
            print("No cells selected.")
            return
        print(f"Selected cells: {len(selected)}")

        new_spd = {}
        for per, recs in spd.items():
            recs = recs.copy()
            mask = []
            for rec in recs:
                i = int(rec["i"])
                j = int(rec["j"])
                mask.append((i, j) in selected)
            mask = np.array(mask, dtype=bool)

            if rate_mode.value == "multiply":
                recs["flux"][mask] *= float(rate_value.value)
            else:
                recs["flux"][mask] = float(rate_value.value)

            if add_missing.value:
                existing = set((int(r["i"]), int(r["j"])) for r in recs)
                to_add = [cell for cell in selected if cell not in existing]
                if to_add:
                    new_recs = np.zeros(len(recs) + len(to_add), dtype=recs.dtype)
                    new_recs[: len(recs)] = recs
                    for idx, (row, col) in enumerate(to_add, start=len(recs)):
                        new_recs[idx]["k"] = int(layer_new.value)
                        new_recs[idx]["i"] = int(row)
                        new_recs[idx]["j"] = int(col)
                        new_recs[idx]["flux"] = float(rate_value.value)
                    recs = new_recs

            new_spd[per] = recs

        wel.stress_period_data = new_spd
        print("Updated WEL stress period data.")

apply_btn.on_click(_apply_update)

display(widgets.VBox([
    mode,
    zone_field,
    zone_values,
    cell_id_text,
    rowcol_text,
    layer_new,
    rate_mode,
    rate_value,
    add_missing,
    apply_btn,
    output,
]))


# Map-based selection (geodatabase + ipyleaflet)

This section loads the `ebfz_b_grid_poly101223` layer from the geodatabase and
lets you click to select cells on a map, then update the WEL data.


In [ ]:
# If needed, install extras
# !pip install ipyleaflet geopandas fiona shapely


In [ ]:
import geopandas as gpd
from ipyleaflet import Map, GeoJSON, LayersControl, WidgetControl
import ipywidgets as widgets
from shapely.geometry import Point
import numpy as np

gdb_path = "ebfz_b_grid/ebfz_b_grid.gdb"
layer_name = "ebfz_b_grid_poly101223"

gdf = gpd.read_file(gdb_path, layer=layer_name)
gdf = gdf.to_crs("EPSG:4326")

# Use centroids for map markers
gdf["_centroid"] = gdf.geometry.centroid
gdf["_lon"] = gdf["_centroid"].x
gdf["_lat"] = gdf["_centroid"].y

center = [float(gdf["_lat"].median()), float(gdf["_lon"].median())]
m = Map(center=center, zoom=9)

# Render polygons
geojson = GeoJSON(data=gdf.drop(columns=["_centroid"]).__geo_interface__, name="Grid")
m.add_layer(geojson)
m.add_control(LayersControl())

selected = set()

def _on_click(event, feature, **kwargs):
    # Toggle selection by CELL_ID
    props = feature.get("properties", {})
    cell_id = int(props.get("CELL_ID"))
    if cell_id in selected:
        selected.remove(cell_id)
    else:
        selected.add(cell_id)
    status.value = f"Selected cells: {len(selected)}"

geojson.on_click(_on_click)

status = widgets.Label(value="Selected cells: 0")
m.add_control(WidgetControl(widget=status, position="topright"))
m


In [ ]:
# Apply updates to the loaded WEL (ensure the WEL load cell has been run)

rate_mode = "multiply"  # or 'set'
rate_value = 1.1
layer_for_new = 1
add_missing = True

spd = wel.stress_period_data.data

# Build map from CELL_ID -> (ROW, COL)
cell_lookup = dict(zip(gdf["CELL_ID"], zip(gdf["ROW"], gdf["COL"])))
selected_cells = {cell_lookup[cid] for cid in selected if cid in cell_lookup}

new_spd = {}
for per, recs in spd.items():
    recs = recs.copy()
    mask = []
    for rec in recs:
        i = int(rec["i"])
        j = int(rec["j"])
        mask.append((i, j) in selected_cells)
    mask = np.array(mask, dtype=bool)

    if rate_mode == "multiply":
        recs["flux"][mask] *= float(rate_value)
    else:
        recs["flux"][mask] = float(rate_value)

    if add_missing and selected_cells:
        existing = set((int(r["i"]), int(r["j"])) for r in recs)
        to_add = [cell for cell in selected_cells if cell not in existing]
        if to_add:
            new_recs = np.zeros(len(recs) + len(to_add), dtype=recs.dtype)
            new_recs[: len(recs)] = recs
            for idx, (row, col) in enumerate(to_add, start=len(recs)):
                new_recs[idx]["k"] = int(layer_for_new)
                new_recs[idx]["i"] = int(row)
                new_recs[idx]["j"] = int(col)
                new_recs[idx]["flux"] = float(rate_value)
            recs = new_recs

    new_spd[per] = recs

wel.stress_period_data = new_spd
print("Updated WEL using map selection.")
